# Knowledge Graph Reasoning

Owner: Member 4

This notebook checks whether linked claims are supported, contradicted, or unknown using conservative knowledge-graph reasoning rules.

## Role in the Project

Notebook 3 links extracted subjects and objects to Wikidata entities. This notebook uses those links to produce a reasoning result for every claim. The output `04_KG_Reasoning/kg_results.json` is the expected input for Bayesian inference in Notebook 5.

In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'LIAR_dataset').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

MODULE_DIR = PROJECT_ROOT / '04_KG_Reasoning'
sys.path.insert(0, str(MODULE_DIR))

from run_kg_reasoning import run_kg_reasoning

INPUT_PATH = PROJECT_ROOT / '03_Entity_Linking_KR' / 'linked_entities.json'
OUTPUT_PATH = MODULE_DIR / 'kg_results.json'
SUMMARY_PATH = MODULE_DIR / 'kg_reasoning_summary.json'

INPUT_PATH, OUTPUT_PATH, SUMMARY_PATH

(PosixPath('/Users/shanusharma/AIaP-Group_Project/03_Entity_Linking_KR/linked_entities.json'),
 PosixPath('/Users/shanusharma/AIaP-Group_Project/04_KG_Reasoning/kg_results.json'),
 PosixPath('/Users/shanusharma/AIaP-Group_Project/04_KG_Reasoning/kg_reasoning_summary.json'))

## Input Preview

The input contains linked subject/object entities, optional Wikidata property IDs, and entity-linking confidence from Notebook 3.

In [2]:
linked_records = json.loads(INPUT_PATH.read_text())
linked_records[:3]

[{'claim_id': 'claim-00001',
  'raw_claim': 'Says the Annies List political group supports third-trimester abortions on demand.',
  'label': 'false',
  'speaker': 'dwayne-bohac',
  'subject': 'Annies List',
  'subject_type': 'ORG',
  'subject_kg_id': None,
  'subject_kg_label': None,
  'subject_kg_description': None,
  'subject_link_source': None,
  'relation': 'say',
  'property_id': None,
  'property_label': None,
  'object': 'demand',
  'object_type': None,
  'object_kg_id': None,
  'object_kg_label': None,
  'object_kg_description': None,
  'object_link_source': None,
  'date': nan,
  'extraction_confidence': 0.8949000239,
  'linking_confidence': 0.224,
  'linking_notes': 'Subject not linked or not suitable for linking. Object not linked or not suitable for linking. No reliable Wikidata property mapping for relation.'},
 {'claim_id': 'claim-00002',
  'raw_claim': 'When did the decline of coal start? It started when natural gas took off that started to begin in (President George W.)

## Reasoning Method

The rules are intentionally conservative:

- use property IDs only when comparison evidence is available
- support or contradict only narrow class claims using Wikidata descriptions
- treat speech relations such as `say` as `unknown`
- treat low-confidence entity links as `unknown`
- preserve every claim so Bayesian inference receives a complete handoff

In [3]:
results = run_kg_reasoning(PROJECT_ROOT)
summary = results['summary']
summary

{'input_path': '03_Entity_Linking_KR/linked_entities.json',
 'output_path': '04_KG_Reasoning/kg_results.json',
 'total_claims': 500,
 'status_counts': {'unknown': 499, 'supported': 1},
 'average_kg_confidence': 0.32,
 'reasoning_rule_counts': [{'reasoning_rule': 'insufficient_kg_evidence',
   'count': 499},
  {'reasoning_rule': 'description_class_match_city', 'count': 1}]}

## Output Preview

Each output record contains the required KG fields: `claim_id`, `kg_status`, `kg_confidence`, `evidence`, `reasoning_rule`, and `source`. Extra context fields are kept for traceability.

In [4]:
kg_df = results['results_df']
kg_df.head()

,claim_id,raw_claim,kg_status,kg_confidence,evidence,reasoning_rule,source,label,subject,subject_kg_id,subject_kg_label,relation,property_id,object,object_kg_id,object_kg_label,linking_confidence
0,claim-00001,Says the Annies List political group supports ...,unknown,0.25,"No subject or object KG entity was linked, so ...",insufficient_kg_evidence,local KG reasoning rules,false,Annies List,NaN,NaN,say,NaN,demand,NaN,NaN,0.224
1,claim-00002,When did the decline of coal start? It started...,unknown,0.40,"Linked entities are available, but no safe KG ...",insufficient_kg_evidence,local KG reasoning rules,half-true,the decline,NaN,NaN,do,NaN,George W.) Bush,Q207,George W. Bush,0.421
2,claim-00003,"Hillary Clinton agrees with John McCain ""by vo...",unknown,0.40,"Linked entities are available, but no safe KG ...",insufficient_kg_evidence,local KG reasoning rules,mostly-true,Hillary Clinton,Q6294,Hillary Clinton,agree,NaN,John McCain,Q10390,John McCain,0.789
3,claim-00004,Health care reform legislation is likely to ma...,unknown,0.25,"No subject or object KG entity was linked, so ...",insufficient_kg_evidence,local KG reasoning rules,false,Health care reform legislation,NaN,NaN,be,NaN,free sex change surgeries,NaN,NaN,0.075
4,claim-00005,The economic turnaround started at the end of ...,unknown,0.25,"No subject or object KG entity was linked, so ...",insufficient_kg_evidence,local KG reasoning rules,half-true,The economic turnaround,NaN,NaN,start,NaN,at the end,NaN,NaN,0.075


## Status Distribution

This table is useful for the empirical analysis section of the report.

In [5]:
kg_df['kg_status'].value_counts()

kg_status
unknown      499
supported      1
Name: count, dtype: int64

## Supported and Contradicted Examples

Because the rules are conservative, most records will remain `unknown`. Any supported or contradicted rows should be inspected before they are used in the report.

In [6]:
kg_df[kg_df['kg_status'].isin(['supported', 'contradicted'])][[
    'claim_id', 'kg_status', 'kg_confidence', 'subject', 'relation', 'object', 'evidence'
]]

,claim_id,kg_status,kg_confidence,subject,relation,object,evidence
36,claim-00037,supported,0.671,Austin,be,a city,Wikidata description for Austin is 'seat of Tr...


## Limitations

The KG reasoning output is limited by the quality of the extracted triples and entity links. Many relations from Notebook 1 are generic verbs such as `say`, `be`, and `have`, so they cannot be safely converted into Wikidata claims. The system therefore favours `unknown` over unsupported support/contradiction decisions.